# Week2_ex1_region_expansion

Reruns the straight-wire H-field exercise with the region's `+Y`/`-Y`/`+Z`/`-Z` padding tripled (`line_length` -> `3*line_length`, i.e. 100mm -> 300mm), keeping `field_line` and everything else identical, to test the region-boundary-proximity hypothesis in `Week2_ex1_analysis.ipynb` Section 9: the original region places the radiation boundary exactly at `field_line`'s far endpoint (y=100mm); if that coincidence is what's driving the growing negative error toward r=100mm, pushing the boundary away should shrink it.


In [ ]:
import ansys.aedt.core
import os


In [ ]:
## Ansys Electronics Desktop (AEDT) interface -- fresh project, not attaching to the original ##

DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)
DT.disable_autosave()

sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

oDesign = M3D.odesign


In [ ]:
## save path -- separate project name so the original Week2_ex1 project is untouched ##

proj_name = "Week2_ex1_region3x"

base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
os.makedirs(dir, exist_ok=True)

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

desi_name = "Week2_ex1_region3x"
M3D.rename_design(desi_name, save=False)
M3D.save_project()


In [ ]:
# same helper as the original notebook -- finds the coil terminal faces by max abs(x) on the winding object
def find_terminal_face(winding_obj) :
    terminal_face = []

    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
## design variables -- identical to the original; the region padding is changed separately below, not through a design variable ##

line_length = 100
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
## setup -- same EddyCurrent rename-bug workaround as the original (raw InsertSetup + fixed setup_name string) ##

oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 10,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 15,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"CacheSaveKind:="	, "Delta",
		"ConstantDelta:="	, "0s",
		"UseCacheFor:="		, ["Freq"],
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, True,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])

setup_name = "Setup1"


In [ ]:
## geometry -- identical to the original except region_tmp: the Y/Z padding is tripled (line_length -> 3*line_length)
## so the radiation boundary sits at y=+-300mm instead of y=+-100mm. field_line is UNCHANGED (still 0-100mm) so the
## exported CSV lines up row-for-row against Week2_ex1_H_field_result.csv. X padding is left as line_length/2
## (matches the wire's own ends) since field_line runs along y, not x. ##

point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                                   xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

region_tmp = ["line_length/2", "-line_length/2", "3*line_length", "-3*line_length", "3*line_length", "-3*line_length"]
region = M3D.modeler.create_region(pad_value=region_tmp ,pad_type="Absolute Position")

assignment = [region.top_face_z, region.bottom_face_z, region.top_face_y, region.bottom_face_y]

# same radiation-boundary workaround as the original -- assign_radiation() rejects "AC Magnetic" solution_type string
oModule = oDesign.GetModule("BoundarySetup")
face_ids = [f.id for f in assignment]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])

box_origin = ["line_diameter", "line_length", "line_length"]
box_sizes = ["-2*line_diameter", "-2*line_length", "-2*line_length"]
dummy = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="dummy", material="vacuum")
dummy.subtract(tool_list=line, keep_originals=True)

point_tmp = []
point_tmp.append(["0mm", "0mm", "0mm"])
point_tmp.append(["0mm", line_length, "0mm"])
field_line = M3D.modeler.create_polyline(points=point_tmp, name="field_line")


In [ ]:
## mesh -- same MaxLength as the original. NOTE: the region is now 3x larger in Y/Z (9x the cross-sectional area of
## the outer shell), so this is the step most likely to hit the Student mesh element cap -- if a cap warning shows up,
## let the run finish before reacting; it can still solve correctly under AEDT's self-capping. Only coarsen MaxLength
## further if it comes back as an actual execution failure. ##

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:line_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5",
	])

oModule.AssignLengthOp(
	[
		"NAME:dummy_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5"
	])


In [ ]:
## excitation -- identical to the original ##

ter_out, ter_in = find_terminal_face(line)

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")

coil = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                            resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                             name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
## analyze ##

oDesign.Analyze(setup_name)


In [ ]:
## export the same field_line report as the original, so it's a direct row-for-row comparison ##

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("Calculator Expressions Plot 1", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:="		, "field_line",
		"PointCount:="		, 301
	], 
	[
		"Distance:="		, ["All"],
		"Freq:="		, ["All"],
		"Phase:="		, ["0deg"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Distance",
		"Y Component:="		, ["Mag_H"]
	])

csv_name = "Week2_ex1_region_expansion_H_field_result.csv"
dir_csv = csv_name
oModule.ExportToFile("Calculator Expressions Plot 1", dir_csv, False)


In [ ]:
## save project ##

M3D.save_project()
